In [ ]:
from typing import Annotated, TypedDict, List, Dict, Any, Optional
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from IPython.display import Image, display
import gradio as gr
import uuid
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

### For structured outputs, we define a Pydantic object for the Schema

In [ ]:
# First define a structured output

class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the assistant's response")
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    user_input_needed: bool = Field(description="True if more input is needed from the user, or clarifications, or the assistant is stuck")


### And for the State, we'll use TypedDict again

But now we have some real information to maintain!

The messages uses the reducer. The others are simply values that we overwrite with any state change.

In [ ]:
# The state

class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool

In [ ]:
# Get our async Playwright tools
# If you get a NotImplementedError here or later, see the Heads Up at the top of the 3_lab3 notebook


import nest_asyncio
nest_asyncio.apply()
async_browser =  create_async_playwright_browser(headless=False)  # headful mode
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
tools = toolkit.get_tools()

In [ ]:
# Initialize the LLMs

worker_llm = ChatOpenAI(model="gpt-4o-mini")
worker_llm_with_tools = worker_llm.bind_tools(tools)

evaluator_llm = ChatOpenAI(model="gpt-4o-mini")
evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)

In [ ]:
# The worker node

def worker(state: State) -> Dict[str, Any]:
    system_message = f"""You are a helpful assistant that can use tools to complete tasks.
You keep working on a task until either you have a question or clarification for the user, or the success criteria is met.
This is the success criteria:
{state['success_criteria']}
You should reply either with a question for the user about this assignment, or with your final response.
If you have a question for the user, you need to reply by clearly stating your question. An example might be:

Question: please clarify whether you want a summary or a detailed answer

If you've finished, reply with the final answer, and don't ask a question; simply reply with the answer.
"""
    
    if state.get("feedback_on_work"):
        system_message += f"""
Previously you thought you completed the assignment, but your reply was rejected because the success criteria was not met.
Here is the feedback on why this was rejected:
{state['feedback_on_work']}
With this feedback, please continue the assignment, ensuring that you meet the success criteria or have a question for the user."""
    
    # Add in the system message

    found_system_message = False
    messages = state["messages"]
    for message in messages:
        if isinstance(message, SystemMessage):
            message.content = system_message
            found_system_message = True
    
    if not found_system_message:
        messages = [SystemMessage(content=system_message)] + messages
    
    # Invoke the LLM with tools
    response = worker_llm_with_tools.invoke(messages)
    
    # Return updated state
    return {
        "messages": [response],
    }

In [ ]:
def worker_router(state: State) -> str:
    last_message = state["messages"][-1]
    
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    else:
        return "evaluator"

In [ ]:
def format_conversation(messages: List[Any]) -> str:
    conversation = "Conversation history:\n\n"
    for message in messages:
        if isinstance(message, HumanMessage):
            conversation += f"User: {message.content}\n"
        elif isinstance(message, AIMessage):
            text = message.content or "[Tools use]"
            conversation += f"Assistant: {text}\n"
    return conversation

In [ ]:
def evaluator(state: State) -> State:
    last_response = state["messages"][-1].content

    system_message = """You are an evaluator that determines if a task has been completed successfully by an Assistant.
Assess the Assistant's last response based on the given criteria. Respond with your feedback, and with your decision on whether the success criteria has been met,
and whether more input is needed from the user."""
    
    user_message = f"""You are evaluating a conversation between the User and Assistant. You decide what action to take based on the last response from the Assistant.

The entire conversation with the assistant, with the user's original request and all replies, is:
{format_conversation(state['messages'])}

The success criteria for this assignment is:
{state['success_criteria']}

And the final response from the Assistant that you are evaluating is:
{last_response}

Respond with your feedback, and decide if the success criteria is met by this response.
Also, decide if more user input is required, either because the assistant has a question, needs clarification, or seems to be stuck and unable to answer without help.
"""
    if state["feedback_on_work"]:
        user_message += f"Also, note that in a prior attempt from the Assistant, you provided this feedback: {state['feedback_on_work']}\n"
        user_message += "If you're seeing the Assistant repeating the same mistakes, then consider responding that user input is required."
    
    evaluator_messages = [SystemMessage(content=system_message), HumanMessage(content=user_message)]

    eval_result = evaluator_llm_with_output.invoke(evaluator_messages)
    new_state = {
        "messages": [{"role": "assistant", "content": f"Evaluator Feedback on this answer: {eval_result.feedback}"}],
        "feedback_on_work": eval_result.feedback,
        "success_criteria_met": eval_result.success_criteria_met,
        "user_input_needed": eval_result.user_input_needed
    }
    return new_state

In [ ]:
def route_based_on_evaluation(state: State) -> str:
    if state["success_criteria_met"] or state["user_input_needed"]:
        return "END"
    else:
        return "worker"

In [ ]:
# Set up Graph Builder with State
graph_builder = StateGraph(State)

# Add nodes
graph_builder.add_node("worker", worker)
graph_builder.add_node("tools", ToolNode(tools=tools))
graph_builder.add_node("evaluator", evaluator)

# Add edges
graph_builder.add_conditional_edges("worker", worker_router, {"tools": "tools", "evaluator": "evaluator"})
graph_builder.add_edge("tools", "worker")
graph_builder.add_conditional_edges("evaluator", route_based_on_evaluation, {"worker": "worker", "END": END})
graph_builder.add_edge(START, "worker")

# Compile the graph
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

### Next comes the gradio Callback to kick off a super-step

In [ ]:
def make_thread_id() -> str:
    return str(uuid.uuid4())


async def process_message(message, success_criteria, history, thread):

    config = {"configurable": {"thread_id": thread}}

    state = {
        "messages": message,
        "success_criteria": success_criteria,
        "feedback_on_work": None,
        "success_criteria_met": False,
        "user_input_needed": False
    }
    result = await graph.ainvoke(state, config=config)
    user = {"role": "user", "content": message}
    reply = {"role": "assistant", "content": result["messages"][-2].content}
    feedback = {"role": "assistant", "content": result["messages"][-1].content}
    return history + [user, reply, feedback]

async def reset():
    return "", "", None, make_thread_id()



### And now launch our Sidekick UI

In [ ]:

with gr.Blocks(theme=gr.themes.Default(primary_hue="emerald")) as demo:
    gr.Markdown("## Sidekick Personal Co-worker")
    thread = gr.State(make_thread_id())
    
    with gr.Row():
        chatbot = gr.Chatbot(label="Sidekick", height=300, type="messages")
    with gr.Group():
        with gr.Row():
            message = gr.Textbox(show_label=False, placeholder="Your request to your sidekick")
        with gr.Row():
            success_criteria = gr.Textbox(show_label=False, placeholder="What are your success critiera?")
    with gr.Row():
        reset_button = gr.Button("Reset", variant="stop")
        go_button = gr.Button("Go!", variant="primary")
    message.submit(process_message, [message, success_criteria, chatbot, thread], [chatbot])
    success_criteria.submit(process_message, [message, success_criteria, chatbot, thread], [chatbot])
    go_button.click(process_message, [message, success_criteria, chatbot, thread], [chatbot])
    reset_button.click(reset, [], [message, success_criteria, chatbot, thread])

    
demo.launch()

In [ ]:


import random

from gradio import themes
class TestSate(TypedDict):
   messages : Annotated[List[Any],add_messages]
   asked : List[Any]
   done:bool

async def clarifier(state: TestSate) -> Dict[str, Any]:

    qstns = ['What is your name?','What is your favourite color','What is your current goal?']
    print(f"Inside clarifier. Question Count: {len(state['asked'])}")
    msg = ""
    if len(state['asked']) < 3:
        while True:
            qstn = qstns[random.randint(0,len(qstns) - 1)]
            asked = state['asked']
            print(f"Qstn:{qstn} \n Asked: {asked}")
            
            if (qstn not in asked):
                asked = asked + [qstn]
                state['asked'] = asked
                break;
            else:
                print(f'Duplicate question {qstn}. Retry')
        msg = {"messages":[{"role": "assistant", "content": f"{qstn}"}]}
    else:
         state['done'] = True
         msg = {"messages":[{"role": "assistant", "content": f"Thank you for answering all the  {len(qstns)} questions"}]}

    print(msg)
    return msg
     
# Set up Graph Builder with State
graph_builder = StateGraph(TestSate)

# Add nodes
graph_builder.add_node("clarifier", clarifier)
#graph_builder.add_node("evaluator", evaluator)

# Add edges
#graph_builder.add_conditional_edges("clarifier", clarifier_router, {"END": "END", "eval": "eval"})
graph_builder.add_edge("clarifier", END)
graph_builder.add_edge(START, "clarifier")

# Compile the graph
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)




async def setup2():
    print("Inside setup")
    
    
    return {"count":0,"last_q":[]}

async def process_m(grState, message, success_criteria, chatbot):
    print(grState)
    print(f"Inside process_m. Gradio state passed. Count: {grState['count']} Last_Q: {grState['last_q']}")
    userMessage = {"role": "user", "content": message}
    config = {"configurable": {"thread_id": "t1"}}

    state = {
        "messages": message,
        "asked": grState['last_q'],
        "done": grState['count'] >= 3
    }
    result = await graph.ainvoke(state, config=config)
    user = {"role": "user", "content": message}
    reply = {"role": "assistant", "content": result["messages"][-1].content}
   
    if(state['done'] == False):
         grState['last_q'] = grState['last_q'] + [reply['content']]
         grState['count'] = len(grState['last_q'])
         reply['content'] = f"Clarifying Question {len(state['asked'])+1} :\n {reply['content']}"
    return chatbot + [user, reply],grState


with gr.Blocks(title="Sidekick", theme=gr.themes.Ocean()) as ui:
    gr.Markdown("## Sidekick Personal Co-Worker")
    grState = gr.State({"count":0,"last_q":[]})
    print(grState)

    with gr.Row():
        chatbot = gr.Chatbot(label="Sidekick", height=300, type="messages",layout="bubble")
    with gr.Group():
        with gr.Row():
            message = gr.Textbox(show_label=False, placeholder="Your request to the Sidekick")
        with gr.Row():
            success_criteria = gr.Textbox(
                show_label=False, placeholder=f"What are your success critiera? {grState.value['count']}"
            )
    with gr.Row():
        
        go_button = gr.Button("Go!", variant="primary")

    ui.load(setup2, [], [grState])
    message.submit(
        process_m, [grState, message, success_criteria , chatbot], [chatbot, grState]
    )
    success_criteria.submit(
       
        process_m, [grState, message, success_criteria, chatbot], [chatbot, grState]
    )
    go_button.click(
        process_m, [grState, message, success_criteria, chatbot], [chatbot, grState]
    )
    


ui.launch(inbrowser=True)

In [ ]:
# -------------------------------------------------
# New cell: Turn-based clarifier that pauses for user input
# -------------------------------------------------
import random
from typing import List, Any, Dict, Annotated, TypedDict
from gradio import themes
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import add_messages   # needed for the annotation

# ------------------------------
# State definition
# ------------------------------
class TestSate(TypedDict):
    messages: Annotated[List[Any], add_messages]   # chat history
    asked: List[Any]                               # questions already asked
    done: bool                                     # true when we asked 3

# ------------------------------
# Constants
# ------------------------------
QSTNS = [
    "What is your name?",
    "What is your favourite colour?",
    "What is your current goal?",
]

# ------------------------------
# Node: ask_question
# ------------------------------
def ask_question(state: TestSate) -> Dict[str, Any]:
    """Pick a random question that hasn't been asked yet."""
    print(f"[ask_question] asked so far: {state['asked']}")
    while True:
        q = QSTNS[random.randint(0, len(QSTNS) - 1)]
        if q not in state['asked']:
            break
        print(f"[ask_question] Duplicate '{q}', retrying...")

    new_asked = state['asked'] + [q]
    return {
        "messages": [{"role": "assistant", "content": q}],
        "asked": new_asked,
    }

# ------------------------------
# Node: finish
# ------------------------------
def finish(state: TestSate) -> Dict[str, Any]:
    """All three questions have been asked – thank the user."""
    print("[finish] All questions asked.")
    return {
        "messages": [
            {
                "role": "assistant",
                "content": f"Thank you for answering all the {len(QSTNS)} questions",
            }
        ],
        "done": True,
    }

# ------------------------------
# Router: decide where to go from START
# ------------------------------
def start_router(state: TestSate) -> str:
    """If we already asked 3 questions, go to finish; otherwise ask another."""
    return "finish" if len(state['asked']) >= len(QSTNS) else "ask_question"

# ------------------------------
# Build the graph
# ------------------------------
graph_builder = StateGraph(TestSate)

graph_builder.add_node("ask_question", ask_question)
graph_builder.add_node("finish", finish)

# Entry point routes to ask_question or finish based on current state
graph_builder.add_conditional_edges(
    START,
    start_router,
    {
        "ask_question": "ask_question",
        "finish": "finish",
    },
)

# Both nodes end after one step so Gradio can show output and wait for user
graph_builder.add_edge("ask_question", END)
graph_builder.add_edge("finish", END)

# Compile with an in-memory checkpointer
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

# ------------------------------
# Gradio helper functions
# ------------------------------
async def setup2():
    print("Inside setup")
    return {"count": 0, "last_q": []}

async def process_m(grState, message, success_criteria, chatbot):
    print(grState)
    print(f"Inside process_m. Count: {grState['count']} Last_Q: {grState['last_q']}")

    user_msg = {"role": "user", "content": message}
    config = {"configurable": {"thread_id": "t1"}}

    state = {
        "messages": [user_msg],
        "asked": grState['last_q'],
    }

    count_before = grState['count']  # <-- save count before graph runs

    # Invoke the graph — it asks exactly one question (or finishes) then ENDs
    result = await graph.ainvoke(state, config=config)

    assistant_msg = result["messages"][-1]
    reply = {"role": "assistant", "content": assistant_msg.content}

    # Always sync the asked list from the graph result so we know where we are
    asked_so_far = result.get("asked", grState['last_q'])
    grState['last_q'] = asked_so_far
    grState['count'] = len(grState['last_q'])

    # Only prefix while we are still in the questioning phase
    if count_before < len(QSTNS):
        reply['content'] = f"Clarifying Question {grState['count']}:\n {reply['content']}"

    return chatbot + [user_msg, reply], grState

# ------------------------------
# Gradio UI
# ------------------------------
with gr.Blocks(title="Sidekick", theme=gr.themes.Ocean()) as ui:
    gr.Markdown("## Sidekick Personal Co-Worker")
    grState = gr.State({"count": 0, "last_q": []})

    with gr.Row():
        chatbot = gr.Chatbot(label="Sidekick", height=300, type="messages", layout="bubble")
    with gr.Group():
        with gr.Row():
            message = gr.Textbox(show_label=False,
                                 placeholder="Your request to the Sidekick")
        with gr.Row():
            success_criteria = gr.Textbox(
                show_label=False,
                placeholder=f"What are your success criteria? {grState.value['count']}"
            )
    with gr.Row():
        go_button = gr.Button("Go!", variant="primary")

    # Wire-up
    ui.load(setup2, [], [grState])
    message.submit(process_m, [grState, message, success_criteria, chatbot],
                   [chatbot, grState])
    success_criteria.submit(process_m, [grState, message, success_criteria, chatbot],
                            [chatbot, grState])
    go_button.click(process_m, [grState, message, success_criteria, chatbot],
                    [chatbot, grState])

ui.launch(inbrowser=True)

In [ ]:
# -------------------------------------------------
# Cell-23: Combined Sidekick with Pre-Task Clarifier
# (Plan A – sequential sub-graphs orchestrated in Gradio)
# -------------------------------------------------
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langchain_core.messages import SystemMessage
import gradio as gr

# Rebuild the MAIN Sidekick graph from cells 8─14 so the node functions are available here
sidekick_builder = StateGraph(State)
sidekick_builder.add_node("worker", worker)
sidekick_builder.add_node("tools", ToolNode(tools=tools))
sidekick_builder.add_node("evaluator", evaluator)

sidekick_builder.add_conditional_edges("worker", worker_router, {"tools": "tools", "evaluator": "evaluator"})
sidekick_builder.add_edge("tools", "worker")
sidekick_builder.add_conditional_edges("evaluator", route_based_on_evaluation, {"worker": "worker", "END": END})
sidekick_builder.add_edge(START, "worker")

sidekick_memory = MemorySaver()
sidekick_graph = sidekick_builder.compile(checkpointer=sidekick_memory)

# The clarifier graph (`graph`) and its helpers (ask_question, finish, QSTNS) are in namespace from cell-21 above.

# ------------------------------------------------------------------------------
# Gradio helpers
# ------------------------------------------------------------------------------
async def setup_combined():
    return {
        "phase": "clarify",           # "clarify" → ask 3 questions, then "work"
        "count": 0,                   # how many clarifying questions asked so far
        "last_q": [],                 # list of questions already asked
        "clarifications": [],         # user's answers (copied from last_q on switch)
        "original_request": None,     # first user message (the real task)
        "original_success_criteria": None,
    }


async def process_combined(grState, message, success_criteria, chatbot):
    """
    Orchestrates two phases:
      1. CLARIFY – turn-based clarifier asks 3 questions one at a time.
      2. WORK    – main Sidekick worker/evaluator loop with clarifications injected.
    """
    user_msg = {"role": "user", "content": message}
    cfg = {"configurable": {"thread_id": "t1"}}

    # ------------------------------------------------------------------
    # PHASE 1 ─ CLARIFY
    # ------------------------------------------------------------------
    if grState.get("phase") == "clarify":
        # On the very first Gradio turn, capture the real request / success criteria
        if grState.get("original_request") is None:
            grState["original_request"] = message
            grState["original_success_criteria"] = success_criteria

        count_before = grState["count"]

        state = {
            "messages": [user_msg],
            "asked": grState["last_q"],
        }
        result = await graph.ainvoke(state, config=cfg)

        assistant_msg = result["messages"][-1]
        reply = {"role": "assistant", "content": assistant_msg.content}

        # Sync checkpointed state into Gradio state
        asked_so_far = result.get("asked", grState["last_q"])
        grState["last_q"] = asked_so_far
        grState["count"] = len(grState["last_q"])

        # Prefix only while we are still asking questions (0, 1, or 2 before invoke)
        if count_before < len(QSTNS):
            reply["content"] = f"Clarifying Question {grState['count']}:\n{reply['content']}"

        # Transition to work phase once all 3 questions are on record
        if grState["count"] >= 3:
            grState["phase"] = "work"
            grState["clarifications"] = grState["last_q"]

        return chatbot + [user_msg, reply], grState

    # ------------------------------------------------------------------
    # PHASE 2 ─ WORK
    # ------------------------------------------------------------------
    else:
        system_context = (
            f"User clarifications:\n"
            + "\n".join(grState.get("clarifications", []))
            + f"\n\nOriginal request: {grState['original_request']}\n"
            f"Success criteria: {grState['original_success_criteria'] or ''}"
        )

        state = {
            "messages": [SystemMessage(content=system_context), user_msg],
            "success_criteria": success_criteria or grState.get("original_success_criteria", ""),
            "feedback_on_work": None,
            "success_criteria_met": False,
            "user_input_needed": False,
        }

        result = await sidekick_graph.ainvoke(state, config=cfg)

        user = {"role": "user", "content": message}
        reply = {"role": "assistant", "content": result["messages"][-2].content}
        feedback = {"role": "assistant", "content": result["messages"][-1].content}

        return chatbot + [user, reply, feedback], grState


# ------------------------------------------------------------------------------
# Gradio UI
# ------------------------------------------------------------------------------
with gr.Blocks(title="Sidekick", theme=gr.themes.Ocean()) as combined_ui:
    gr.Markdown("## Sidekick Personal Co-Worker (with Pre-Task Clarifier)")

    grState = gr.State({
        "phase": "clarify",
        "count": 0,
        "last_q": [],
        "clarifications": [],
        "original_request": None,
        "original_success_criteria": None,
    })

    with gr.Row():
        chatbot = gr.Chatbot(label="Sidekick", height=300, type="messages", layout="bubble")

    with gr.Group():
        with gr.Row():
            message = gr.Textbox(show_label=False, placeholder="Your request to the Sidekick")
        with gr.Row():
            success_criteria = gr.Textbox(
                show_label=False, placeholder="What are your success criteria?"
            )

    with gr.Row():
        go_button = gr.Button("Go!", variant="primary")

    # Wiring
    combined_ui.load(setup_combined, [], [grState])
    message.submit(process_combined, [grState, message, success_criteria, chatbot], [chatbot, grState])
    success_criteria.submit(process_combined, [grState, message, success_criteria, chatbot], [chatbot, grState])
    go_button.click(process_combined, [grState, message, success_criteria, chatbot], [chatbot, grState])

# Launch
combined_ui.launch(inbrowser=True)